In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    if stop_sequences is None:
        stop_sequences = []

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system is not None:
        params["system"] = system

    response = client.messages.create(**params)  # ty:ignore[no-matching-overload]
    return response.content[0].text

In [11]:
import json
from textwrap import dedent


def generate_dataset():
    prompt = dedent("""
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """)

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [17]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = dedent(f"""
        Please solve the following task:
        
        {test_case["task"]}
    """)

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [18]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO: Grade logic
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [ ]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


In [21]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Parse S3 Bucket Name\n\nHere's a clean and robust solution:\n\n```python\ndef parse_s3_bucket_name(s3_uri):\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI string.\n    \n    Args:\n        s3_uri (str): S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not s3_uri:\n        raise ValueError(\"S3 URI cannot be empty\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Invalid S3 URI: bucket name cannot be empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n    